# MQ3 - Transformer를 GPT로 변경하여 구현

## 1. Transformer와 비교해 변경이 필요한 부분

이번 과제에서는 기존 Transformer 번역 모델을 GPT 형태의 decoder-only 생성 모델로 변경하였다.

### (1) 전체 아키텍처 변경
- 기존 Transformer는 **Encoder + Decoder** 구조이며, source sentence와 target sentence를 각각 입력받는다.
- GPT는 **Decoder-only Transformer** 구조이므로 Encoder를 제거하고, 하나의 토큰 시퀀스를 입력으로 받아 다음 토큰을 예측하도록 변경하였다.

### (2) Attention 블록 변경
- 기존 Transformer Decoder에는
  1. masked self-attention
  2. encoder-decoder attention
  3. feed-forward network
  가 포함된다.
- GPT는 encoder가 없으므로 **encoder-decoder attention 블록을 제거**하고,
  **masked self-attention + feed-forward network**만 반복하도록 수정하였다.

### (3) 입력 형태 변경
- 기존 번역 모델은 `(enc_in, dec_in)` 형태의 병렬 입력을 사용했다.
- GPT는 언어모델이므로 하나의 연속된 시퀀스만 입력받는다.
- 챗봇 데이터는 질문/답변 쌍을 하나의 문자열 시퀀스로 합쳐서
  `"<usr> 질문 <sys> 답변"` 형태로 만들고,
  입력은 `seq[:-1]`, 정답은 `seq[1:]`가 되도록 변형하였다.

### (4) Positional Encoding 변경
- 기존 Transformer 노드에서는 **sinusoidal positional encoding**을 사용했다.
- GPT 논문에서는 **learned positional embedding**을 사용하므로, 위치 정보를 Embedding 레이어로 학습하도록 변경하였다.

### (5) 학습 목적 변경
- 기존 Transformer는 번역을 위한 seq2seq 학습을 수행한다.
- GPT는 **causal language modeling**을 수행하므로,
  현재 위치 이전 토큰들만 참고하여 다음 토큰을 예측하도록 causal mask를 적용하였다.

### (6) 출력 방식 변경
- 기존 Transformer는 decoder 출력으로 target vocabulary 분포를 예측한다.
- GPT도 각 시점의 vocabulary 분포를 출력하지만,
  하나의 연속 시퀀스에 대해 다음 토큰을 예측하는 방식으로 사용한다.
- 또한 GPT 논문에 맞춰 **token embedding weight와 output projection weight를 공유(weight tying)** 하였다.

In [1]:
# 라이브러리 import
import os
import re
import math
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("torch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch version: 2.7.1+cu118
device: cuda


In [2]:
# sentencepiece 설치 확인
try:
    import sentencepiece as spm
    print("sentencepiece already installed")
except ImportError:
    !pip install sentencepiece
    import sentencepiece as spm

sentencepiece already installed


In [5]:
# 데이터 경로 찾기
def find_chatbot_csv():
    candidate_paths = [
        os.path.join(os.getenv("HOME"), "aiffel", "transformer_chatbot", "data", "ChatbotData.csv"),
        os.path.join(os.getenv("HOME"), "work", "data", "ChatbotData.csv"),
        os.path.join(os.getenv("HOME"), "work", "transformer", "data", "ChatbotData.csv"),
        os.path.join(os.getcwd(), "ChatbotData.csv"),
        "/mnt/data/ChatbotData.csv",
    ]
    
    for path in candidate_paths:
        if os.path.exists(path):
            return path
    return None

data_path = find_chatbot_csv()
print("found path:", data_path)

# 못 찾으면 여기 직접 수정
# data_path = "/home/jovyan/work/.../ChatbotData.csv"

if data_path is None:
    raise FileNotFoundError("ChatbotData.csv를 찾지 못했습니다. data_path를 직접 지정해 주세요.")

found path: /home/jovyan/work/project/first-repository/Main_Quest/Quest03/ChatbotData.csv


In [6]:
# 데이터 로드 및 기본 확인
df = pd.read_csv(data_path)
print(df.head())
print(df.columns)
print("data size:", len(df))

                 Q            A  label
0           12시 땡!   하루가 또 가네요.      0
1      1지망 학교 떨어졌어    위로해 드립니다.      0
2     3박4일 놀러가고 싶다  여행은 언제나 좋죠.      0
3  3박4일 정도 놀러가고 싶다  여행은 언제나 좋죠.      0
4          PPL 심하네   눈살이 찌푸려지죠.      0
Index(['Q', 'A', 'label'], dtype='object')
data size: 11823


In [8]:
# 보통 컬럼명이 Q, A일 가능성이 커서 그대로 사용
assert "Q" in df.columns and "A" in df.columns, "ChatbotData.csv에 Q, A 컬럼이 있어야 합니다."

In [9]:
# 텍스트 전처리
def preprocess_sentence(sentence):
    sentence = str(sentence).strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"[^0-9a-zA-Z가-힣?.!,]+", " ", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence

df["Q"] = df["Q"].apply(preprocess_sentence)
df["A"] = df["A"].apply(preprocess_sentence)

df = df.dropna().drop_duplicates(subset=["Q", "A"]).reset_index(drop=True)
print("cleaned size:", len(df))
df.head()

cleaned size: 11749


,Q,A,label
0,12시 땡 !,하루가 또 가네요 .,0
1,1지망 학교 떨어졌어,위로해 드립니다 .,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠 .,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠 .,0
4,PPL 심하네,눈살이 찌푸려지죠 .,0


In [10]:
# GPT용 데이터 형태로 변환
# [변경점] Transformer의 enc/dec 쌍 입력 대신,
# GPT에서는 하나의 연속된 시퀀스로 만들어 다음 토큰 예측을 수행한다.
# 형식: <usr> 질문 <sys> 답변

USER_TOKEN = "<usr>"
SYS_TOKEN = "<sys>"

corpus = []
for q, a in zip(df["Q"], df["A"]):
    text = f"{USER_TOKEN} {q} {SYS_TOKEN} {a}"
    corpus.append(text)

print(corpus[:3])
print("num samples:", len(corpus))

['<usr> 12시 땡 ! <sys> 하루가 또 가네요 .', '<usr> 1지망 학교 떨어졌어 <sys> 위로해 드립니다 .', '<usr> 3박4일 놀러가고 싶다 <sys> 여행은 언제나 좋죠 .']
num samples: 11749


In [12]:
# SentencePiece 토크나이저 학습
VOCAB_SIZE = 8000
PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3

SPM_PREFIX = "gpt_chatbot_spm"
SPM_INPUT_TXT = "gpt_chatbot_corpus.txt"

with open(SPM_INPUT_TXT, "w", encoding="utf-8") as f:
    for line in corpus:
        f.write(line + "\n")

if not os.path.exists(f"{SPM_PREFIX}.model"):
    spm.SentencePieceTrainer.Train(
        f"--input={SPM_INPUT_TXT} "
        f"--model_prefix={SPM_PREFIX} "
        f"--vocab_size={VOCAB_SIZE} "
        f"--pad_id={PAD_ID} "
        f"--bos_id={BOS_ID} "
        f"--eos_id={EOS_ID} "
        f"--unk_id={UNK_ID} "
        f"--model_type=unigram "
        f"--character_coverage=0.9995 "
        f"--user_defined_symbols={USER_TOKEN},{SYS_TOKEN}"
    )

tokenizer = spm.SentencePieceProcessor()
tokenizer.Load(f"{SPM_PREFIX}.model")

print("vocab size:", tokenizer.GetPieceSize())
print("pad/bos/eos/unk:", PAD_ID, BOS_ID, EOS_ID, UNK_ID)
print("sample encode:", tokenizer.EncodeAsIds(corpus[0])[:30])
print("sample decode:", tokenizer.DecodeIds(tokenizer.EncodeAsIds(corpus[0])[:30]))

vocab size: 8000
pad/bos/eos/unk: 0 1 2 3
sample encode: [6, 4, 4175, 321, 2363, 58, 6, 5, 263, 9, 113, 4543, 8, 7]
sample decode: <usr> 12시 땡 ! <sys> 하루가 또 가네요 .


sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=gpt_chatbot_corpus.txt --model_prefix=gpt_chatbot_spm --vocab_size=8000 --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 --model_type=unigram --character_coverage=0.9995 --user_defined_symbols=<usr>,<sys>
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: gpt_chatbot_corpus.txt
  input_format: 
  model_prefix: gpt_chatbot_spm
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <usr>
  user_defined_symbols: <sys>
  required_chars

In [13]:
# 토큰화 + 길이 제한 + 입력/라벨 생성
MAX_LEN = 64  # 노트북 자원 고려하여 적당히 설정

tokenized_sequences = []
for text in corpus:
    ids = [BOS_ID] + tokenizer.EncodeAsIds(text) + [EOS_ID]
    if len(ids) <= MAX_LEN:
        tokenized_sequences.append(ids)

print("usable samples:", len(tokenized_sequences))
print("sample:", tokenized_sequences[0][:30])

usable samples: 11749
sample: [1, 6, 4, 4175, 321, 2363, 58, 6, 5, 263, 9, 113, 4543, 8, 7, 2]


In [14]:
# Dataset / DataLoader
class GPTDataset(Dataset):
    def __init__(self, sequences, pad_id=0, max_len=64):
        self.sequences = sequences
        self.pad_id = pad_id
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        
        # GPT LM:
        # input_ids = [BOS, w1, w2, ..., wn-1]
        # labels    = [w1,  w2, w3, ..., EOS]
        input_ids = seq[:-1]
        labels = seq[1:]
        
        input_ids = input_ids[:self.max_len-1]
        labels = labels[:self.max_len-1]
        
        input_ids = input_ids + [self.pad_id] * ((self.max_len - 1) - len(input_ids))
        labels = labels + [self.pad_id] * ((self.max_len - 1) - len(labels))
        
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

dataset = GPTDataset(tokenized_sequences, pad_id=PAD_ID, max_len=MAX_LEN)

train_size = int(len(dataset) * 0.9)
valid_size = len(dataset) - train_size
train_dataset, valid_dataset = torch.utils.data.random_split(
    dataset, [train_size, valid_size],
    generator=torch.Generator().manual_seed(SEED)
)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("train size:", len(train_dataset))
print("valid size:", len(valid_dataset))

train size: 10574
valid size: 1175


## 2. GPT 모델 구현

In [15]:
# 마스크 함수
def create_padding_mask(seq, pad_id=0):
    # 1: keep, 0: mask
    return (seq != pad_id).unsqueeze(1).unsqueeze(2)   # (B,1,1,L)

def create_causal_mask(seq_len, device):
    # 하삼각행렬: 현재 위치까지만 보게 함
    return torch.tril(torch.ones((seq_len, seq_len), dtype=torch.bool, device=device)).unsqueeze(0).unsqueeze(0)
    # (1,1,L,L)

def create_gpt_mask(seq, pad_id=0):
    # [변경점] GPT는 decoder-only이므로 causal mask가 필수
    # padding mask + causal mask를 결합하여 미래 토큰을 보지 못하게 만든다.
    B, L = seq.size()
    padding_mask = create_padding_mask(seq, pad_id=pad_id).to(seq.device)   # (B,1,1,L)
    causal_mask = create_causal_mask(L, seq.device)                         # (1,1,L,L)
    mask = padding_mask & causal_mask                                       # broadcast
    return mask

In [16]:
# Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.depth = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        bsz, seq_len, d_model = x.size()
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        return x.permute(0, 2, 1, 3)  # (B,H,L,D)

    def combine_heads(self, x):
        bsz, num_heads, seq_len, depth = x.size()
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.view(bsz, seq_len, self.d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.depth)  # (B,H,L,L)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        return out, attn

    def forward(self, x, mask=None):
        # [변경점] GPT block에서는 self-attention만 사용한다.
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))
        
        out, attn = self.scaled_dot_product_attention(Q, K, V, mask)
        out = self.combine_heads(out)
        out = self.out_proj(out)
        return out, attn

In [17]:
# Feed Forward
    # GPT 논문 쪽은 GELU를 사용한다.
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()  # [변경점] GPT 논문에 맞춰 GELU 사용

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [18]:
# GPT Block
class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model, eps=1e-5)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout=dropout)
        self.dropout1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(d_model, eps=1e-5)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout=dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-LN
        residual = x
        out = self.ln1(x)
        out, attn = self.attn(out, mask=mask)
        out = self.dropout1(out)
        x = residual + out

        residual = x
        out = self.ln2(x)
        out = self.ffn(out)
        out = self.dropout2(out)
        x = residual + out

        return x, attn

In [19]:
# GPT Model
    # Encoder 제거
    # Decoder-only block 반복
    # Learned positional embedding
    # Weight tying
class GPTModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_len,
        n_layers=6,
        d_model=256,
        n_heads=8,
        d_ff=1024,
        dropout=0.1,
        pad_id=0
    ):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.d_model = d_model
        self.pad_id = pad_id
        
        # [변경점] Transformer의 sinusoidal positional encoding 대신
        # GPT 논문에 맞춰 learned positional embedding 사용
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

        # [변경점] Encoder/Decoder 대신 decoder-only block만 반복
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, n_heads, d_ff, dropout=dropout)
            for _ in range(n_layers)
        ])
        
        self.final_ln = nn.LayerNorm(d_model, eps=1e-5)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # [변경점] GPT 방식의 weight tying
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, input_ids):
        B, L = input_ids.size()
        assert L <= self.max_len, f"sequence length {L} > max_len {self.max_len}"
        
        positions = torch.arange(0, L, device=input_ids.device).unsqueeze(0).expand(B, L)
        
        # [변경점] token embedding + learned position embedding
        x = self.token_embedding(input_ids) * math.sqrt(self.d_model)
        x = x + self.position_embedding(positions)
        x = self.dropout(x)

        mask = create_gpt_mask(input_ids, pad_id=self.pad_id)

        attentions = []
        for block in self.blocks:
            x, attn = block(x, mask=mask)
            attentions.append(attn)

        x = self.final_ln(x)
        logits = self.lm_head(x)  # (B,L,V)
        
        return logits, attentions

In [20]:
# 모델 생성 및 출력 확인
model = GPTModel(
    vocab_size=tokenizer.GetPieceSize(),
    max_len=MAX_LEN - 1,   # input 길이 기준
    n_layers=6,
    d_model=256,
    n_heads=8,
    d_ff=1024,
    dropout=0.1,
    pad_id=PAD_ID
).to(device)

print(model)

GPTModel(
  (token_embedding): Embedding(8000, 256, padding_idx=0)
  (position_embedding): Embedding(63, 256)
  (dropout): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-5): 6 x GPTBlock(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=True)
        (W_k): Linear(in_features=256, out_features=256, bias=True)
        (W_v): Linear(in_features=256, out_features=256, bias=True)
        (out_proj): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (dropout1): Dropout(p=0.1, inplace=False)
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ffn): PositionwiseFeedForward(
        (fc1): Linear(in_features=256, out_features=1024, bias=True)
        (fc2): Linear(in_features=1024, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (act): GELU(app

In [21]:
# 입력 블록 정상 구성 확인
    # 평가기준 3 대응 확인
sample_batch = next(iter(train_loader))
sample_input = sample_batch["input_ids"][:2].to(device)

logits, attns = model(sample_input)

print("sample_input shape:", sample_input.shape)
print("logits shape:", logits.shape)
print("num attention blocks:", len(attns))
print("attention shape of block 0:", attns[0].shape)  # (B, H, L, L)

print("\n[decoded sample input]")
for i in range(2):
    ids = sample_input[i].detach().cpu().tolist()
    ids = [x for x in ids if x != PAD_ID]
    print(i, tokenizer.DecodeIds(ids))

sample_input shape: torch.Size([2, 63])
logits shape: torch.Size([2, 63, 8000])
num attention blocks: 6
attention shape of block 0: torch.Size([2, 8, 63, 63])

[decoded sample input]
0 <usr> 남자친구가 너무 바빠 <sys> 너무 집착하지 마세요 .
1 <usr> 의욕이 안 생겨 <sys> 하고 싶은 걸 찾아보세요 .


## 3. 학습

In [22]:
# loss / optimizer
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.01)

In [23]:
# 학습 함수
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    
    for batch in tqdm(dataloader, leave=False):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits, _ = model(input_ids)  # (B, L, V)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
    
    return total_loss / len(dataloader)

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    
    for batch in tqdm(dataloader, leave=False):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        logits, _ = model(input_ids)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1)
        )
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

In [24]:
# 학습 실행
EPOCHS = 10

best_valid_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    valid_loss = evaluate(model, valid_loader, criterion, device)
    
    print(f"[Epoch {epoch:02d}] train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f}")
    
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "best_gpt_chatbot.pt")
        print(">> best model saved")

  0%|          | 0/166 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

[Epoch 01] train_loss=5.2227 | valid_loss=4.4038
>> best model saved


  0%|          | 0/166 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

...

  0%|          | 0/19 [00:00<?, ?it/s]

[Epoch 09] train_loss=3.0218 | valid_loss=3.5736
>> best model saved


  0%|          | 0/166 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

[Epoch 10] train_loss=2.9224 | valid_loss=3.5401
>> best model saved


In [25]:
# 학습된 모델 로드
model.load_state_dict(torch.load("best_gpt_chatbot.pt", map_location=device))
model.eval()
print("best model loaded")

best model loaded


## 4. 생성 함수

In [26]:
@torch.no_grad()
def generate_response(
    model,
    tokenizer,
    question,
    max_new_tokens=30,
    temperature=1.0,
    top_k=20
):
    model.eval()
    
    question = preprocess_sentence(question)
    prompt = f"{USER_TOKEN} {question} {SYS_TOKEN}"
    input_ids = [BOS_ID] + tokenizer.EncodeAsIds(prompt)
    
    for _ in range(max_new_tokens):
        cur_input = input_ids[-(MAX_LEN-1):]
        x = torch.tensor(cur_input, dtype=torch.long, device=device).unsqueeze(0)

        logits, _ = model(x)
        next_token_logits = logits[0, -1, :] / temperature
        
        if top_k is not None:
            values, indices = torch.topk(next_token_logits, k=top_k)
            probs = torch.zeros_like(next_token_logits).scatter_(0, indices, F.softmax(values, dim=-1))
        else:
            probs = F.softmax(next_token_logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1).item()

        if next_token == EOS_ID:
            break
        
        input_ids.append(next_token)
    
    decoded = tokenizer.DecodeIds(input_ids)
    
    # <sys> 이후만 잘라서 답변으로 사용
    if SYS_TOKEN in decoded:
        answer = decoded.split(SYS_TOKEN, 1)[1].strip()
    else:
        answer = decoded.strip()
        
    return answer

In [27]:
# 테스트
test_questions = [
    "안녕",
    "오늘 너무 피곤해",
    "너무 우울해",
    "배가 고파",
    "공부가 하기 싫어"
]

for q in test_questions:
    ans = generate_response(model, tokenizer, q)
    print(f"Q: {q}")
    print(f"A: {ans}")
    print("-" * 60)

Q: 안녕
A: 드드립니다 !
------------------------------------------------------------
Q: 오늘 너무 피곤해
A: 다시 이별하고 하지 말고 그래지 마세요 .
------------------------------------------------------------
Q: 너무 우울해
A: 저도요 .
------------------------------------------------------------
Q: 배가 고파
A: 눈은 스트레스가 좋아요 .
------------------------------------------------------------
Q: 공부가 하기 싫어
A: 다른 사람과 함께 해보세요 .
------------------------------------------------------------


In [28]:
# 테스트2
test_questions = [
    "점심 메뉴 추천해줘",
    "AI 공부를 잘 할 수 있는 방법이 궁금해",
    "오늘은 금요일이야",
    "지금 너의 기분은 어때?",
    "여행 가고 싶어"
]

for q in test_questions:
    ans = generate_response(model, tokenizer, q)
    print(f"Q: {q}")
    print(f"A: {ans}")
    print("-" * 60)

Q: 점심 메뉴 추천해줘
A: 공부는 안 좋아요
------------------------------------------------------------
Q: AI 공부를 잘 할 수 있는 방법이 궁금해
A: 그런 생각을 해보세요 .
------------------------------------------------------------
Q: 오늘은 금요일이야
A: 한 잔 하세요 .
------------------------------------------------------------
Q: 지금 너의 기분은 어때?
A: 당신의 가능성을 수 있어요 .
------------------------------------------------------------
Q: 여행 가고 싶어
A: 가서 집에서 로로 하세요 .
------------------------------------------------------------


## 결과 정리

- 기존 Transformer 번역 모델에서 **Encoder를 제거**하고 **Decoder-only GPT 구조**로 변경하였다.
- 챗봇 데이터의 질문/답변 쌍을 `"<usr> 질문 <sys> 답변"` 형태의 **하나의 연속 시퀀스**로 변환하여 language modeling 학습이 가능하도록 전처리하였다.
- 기존 sinusoidal positional encoding 대신 **learned positional embedding**을 적용하였다.
- **causal mask**를 사용하여 미래 토큰을 보지 못하도록 만들었고, GPT 방식의 **next token prediction**으로 학습하였다.
- 학습 후 입력 질문에 대해 답변 토큰이 생성되는 것을 확인하였다.

### Q&A
왜 encoder를 없앴는가
GPT는 seq2seq 번역 모델이 아니라, 앞 토큰들만 보고 다음 토큰을 맞히는 decoder-only language model이기 때문.

왜 Q/A를 하나로 합쳤는가
GPT는 하나의 연속 시퀀스를 학습하므로
질문, 답변을 따로 넣지 않고
"<usr> 질문 <sys> 답변" 형태의 한 문장으로 만든다.

왜 causal mask를 쓰는가
현재 시점에서 미래 답을 미리 보면 language modeling이 아니므로
현재 이전 토큰만 보도록 제한해야 한다.

왜 learned positional embedding을 썼는가
GPT 논문이 sinusoidal 대신 학습형 위치 임베딩을 사용했기 때문.

왜 encoder-decoder attention이 사라졌는가
encoder output 자체가 없으니 cross-attention 블록도 필요 없어짐.